# **Fine-Tuning a Sentiment Analysis LLM - Llama 2**
**Using a specific dataset with QLORA strategy**

# 1 - Installing required packages

In [1]:
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.4 MB/s eta 0:00:00


In [2]:
!pip install -q accelerate==1.9.0 peft==0.16.0 bitsandbytes==0.46.1 transformers==4.54.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.5 MB/s eta 0:00:00


In [3]:
!pip install -q trl==0.20.0 gradio==5.38.2 protobuf scipy==1.16.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 113.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.


In [4]:
!pip install -q sentencepiece==0.2.0 tokenizers==0.21.2 datasets==4.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.7 MB/s eta 0:00:00


In [5]:
# Imports
import os
import torch
import datasets
import pandas as pd
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser,
                          TrainingArguments, pipeline, logging)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings("ignore")

# 2 - Add configuration to ease development

In [6]:
# Defining logging verbosity
logging.set_verbosity(logging.CRITICAL)

In [7]:
# Checking GPU model available
if torch.cuda.is_available():
  print("GPUs: ", torch.cuda.device_count())
  print("GPU Name: ", torch.cuda.get_device_name(0))
  print("GPU Total Memory: ", torch.cuda.get_device_properties(0).total_memory/1e9)

GPUs:  1
GPU Name:  NVIDIA A100-SXM4-40GB
GPU Total Memory:  42.405855232


In [8]:
# Adding bckp block in order to reset GPU memory, if necessary
from numba import cuda
device = cuda.get_current_device()
device.reset()

In [9]:
!ls drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/

 BKP
'Cópia de FINE_TUNING_SENTIMENT_ANALYSIS_LLM_LLAMA2.ipynb'
 dataset.csv
 FINE_TUNING_SENTIMENT_ANALYSIS_LLM_LLAMA2.ipynb
 saida


# 3 - Loading dataset for Fine-Tuning

In [10]:
# Definig dataset var
dataset_name = 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/dataset.csv'

In [11]:
# Load dataset
## --> The 'load_dataset' is excelent because it will convert the data in a format that LLM can understand
dataset_loaded = load_dataset('csv', data_files = dataset_name, delimiter = ',')

Generating train split: 0 examples [00:00, ? examples/s]

In [12]:
# Show dataset
print(dataset_loaded)

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 17057
    })
})


Let's inspect a few examples from your loaded dataset to understand its structure. This will help us determine if the data is formatted in a way that encourages the model to generate sentiment labels.

In [13]:
# Print the first 5 examples from the 'train' split of the dataset
for i in range(5):
    print(dataset_loaded['train'][i])

{'train': "<s>[INST] The only thing I can write as an average Joe who watches movies simply to be entertained or see a story being told, is that I watched 12 Men argue in a room for more than an hour and I couldn't look away from the screen. Simply a masterpiece! [/INST] Positive </s>"}
{'train': '<s>[INST] Shot in real time, this story of a jury trying to decide whether a young man is innocent or guilty never lets the viewer\'s attention go. Henry Fonda plays Juror #8, who is convinced the entire time that the defendant is innocent, while everyone else has already called him guilty. Throughout the movie, we not only get to hear everyone\'s opinions on the matter, but also every possibility of the verdict. With "12 Angry Men", Sidney Lumet brought to the screen the same kind of "closing in" feeling that he brought to movies like "Network". In the end, the issue is not whether the defendant is innocent or guilty; the important point is the process by which the jurors reached their decis

# 4 - Defining params before Fine-Tuning

In [14]:
# Defining LORA params
lora_r = 32
lora_alpha = 16
lora_dropout = 0.1

In [15]:
# Defining QLORA params
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

In [16]:
# Defining Fine-Tuning params
output_dir = "drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "adamw_8bit"
lr_scheduler_type = "linear"
max_steps = -1
warmup_ratio = 0.03
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
seed = 3407
report_to = "none"
dataset_text_field = "train"
packing = False

In [17]:
# Defining extra fine-tuning params
## --> Related to batch padding, saving and logging
group_by_length = True
save_steps = 0
logging_steps = 400

# 5 - Fine-Tuning the LLM

5.1 - Training

In [18]:
# Defining extra important vars
repository_hf = "NousResearch/Llama-2-7b-chat-hf"
model_tuned = "drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned"

In [19]:
# Defining LORA config
peft_config = LoraConfig(lora_alpha = lora_alpha,
                         lora_dropout = lora_dropout,
                         r = lora_r,
                         bias = "none",
                         task_type = "CAUSAL_LM")

In [20]:
# Defining QLORA (quantization) config
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [21]:
# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained(repository_hf,
                                              quantization_config = bnb_config,
                                              device_map = "auto")

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [22]:
# Dont need for cache or parallelism
model.config.use_cache = False
model.config.pretraining_tp = 1

In [23]:
# Loading tokenizer of pre-trained model
tokenizer = AutoTokenizer.from_pretrained(repository_hf, trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [24]:
# Defining traning params (1º version)
training_arguments = TrainingArguments(output_dir = output_dir,
                                       num_train_epochs = num_train_epochs,
                                       per_device_train_batch_size = per_device_train_batch_size,
                                       gradient_accumulation_steps = gradient_accumulation_steps,
                                       optim = optim,
                                       save_steps = save_steps,
                                       logging_steps = logging_steps,
                                       learning_rate = learning_rate,
                                       weight_decay = weight_decay,
                                       fp16 = fp16,
                                       bf16 = bf16,
                                       max_grad_norm = max_grad_norm,
                                       max_steps = max_steps,
                                       warmup_ratio = warmup_ratio,
                                       group_by_length = group_by_length,
                                       lr_scheduler_type = lr_scheduler_type
                      )

In [25]:
# Defining traning params (2º version)
sft_config = SFTConfig(per_device_train_batch_size = per_device_train_batch_size,
                       gradient_accumulation_steps = gradient_accumulation_steps,
                       warmup_steps = 5,
                       num_train_epochs = num_train_epochs,
                       learning_rate = learning_rate,
                       fp16 = fp16,
                       logging_steps = logging_steps,
                       optim = optim,
                       weight_decay = weight_decay,
                       lr_scheduler_type = lr_scheduler_type,
                       seed = seed,
                       output_dir = output_dir,
                       report_to = report_to,
                       dataset_text_field = dataset_text_field,
                       packing = packing
            )

In [26]:
# Creating the trainer object
model_trainer = SFTTrainer(
                            model = model,
                            train_dataset = dataset_loaded["train"],
                            peft_config = peft_config,
                            args = sft_config
                )

Adding EOS to train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

In [27]:
# Training the fine-tuned model
%%time
model_trainer.train()

{'loss': 0.1707, 'grad_norm': nan, 'learning_rate': 4e-05, 'num_tokens': 186549.0, 'mean_token_accuracy': 0.23496039687655867, 'epoch': 0.09378663540445487}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4e-05, 'num_tokens': 366230.0, 'mean_token_accuracy': 0.22510073438286782, 'epoch': 0.18757327080890973}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4e-05, 'num_tokens': 551787.0, 'mean_token_accuracy': 0.2268823660351336, 'epoch': 0.2813599062133646}
{'loss': 0.0154, 'grad_norm': nan, 'learning_rate': 8e-05, 'num_tokens': 748821.0, 'mean_token_accuracy': 0.2297710788808763, 'epoch': 0.37514654161781946}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 8e-05, 'num_tokens': 926601.0, 'mean_token_accuracy': 0.22829720189794897, 'epoch': 0.46893317702227433}
{'loss': 0.2563, 'grad_norm': nan, 'learning_rate': 0.00012, 'num_tokens': 1111834.0, 'mean_token_accuracy': 0.22902904821559786, 'epoch': 0.5627198124267292}
{'loss': 0.0265, 'grad_norm': nan, 'learning_rate': 0.00016, 'num_to

TrainOutput(global_step=4265, training_loss=0.07874706487443216, metrics={'train_runtime': 1795.4767, 'train_samples_per_second': 9.5, 'train_steps_per_second': 2.375, 'total_flos': 1.6463087807493734e+17, 'train_loss': 0.07874706487443216})

In [28]:
# Saving tuned model
model_trainer.model.save_pretrained(model_tuned)

# 6 - Generating responses using the Fine-Tuned LLM

In [29]:
# Generating a prompt
prompt = "I feel like trying to cope with AI news an impossible task nowadays"

In [30]:
# Generating a pipe
pipe = pipeline(task = "text-generation",
                model = model,
                tokenizer = tokenizer,
                max_length = 200)

In [31]:
# Executing the pipeline and extract result
# Update the prompt to explicitly ask for the sentiment
prompt_for_model = f"<s>[INST] Analyze the sentiment of the following text: '{prompt}'. Is it Positive or Negative? [/INST]"
result = pipe(prompt_for_model, max_new_tokens=10) # Limit generation length to just get the label
print(result[0]["generated_text"])

<s>[INST] Analyze the sentiment of the following text: 'I feel like trying to cope with AI news an impossible task nowadays'. Is it Positive or Negative? [/INST]  The sentiment of the text is Negative.


# 7 - Saving Model for Deployment

In [32]:
# Saving the tokenizer (final)
tokenizer_final = AutoTokenizer.from_pretrained(repository_hf, trust_remote_code = True)
tokenizer_final.pad_token = tokenizer_final.eos_token
tokenizer_final.padding_side = "right"
tokenizer_final.save_pretrained(model_tuned+"/final/tokenizer_final")

('drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned/final/tokenizer_final/tokenizer_config.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned/final/tokenizer_final/special_tokens_map.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned/final/tokenizer_final/tokenizer.model',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned/final/tokenizer_final/added_tokens.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/model_tuned/final/tokenizer_final/tokenizer.json')

In [33]:
# Saving the model (final)
model_tuned_final = PeftModel.from_pretrained(model, model_tuned)
model_tuned_final = model_tuned_final.merge_and_unload()
model_tuned_final.save_pretrained(model_tuned+"/final/model_tuned_final")